# Day 5 Function Guide

This notebook documents the data-loading, validation, and target-construction functions added for Day 5 of the NYC weather bucket-market prototype.

The Day 5 code does not train a model and does not engineer Day 6 features. It only loads Open-Meteo CSV data, cleans a few core columns, validates the data shape, and constructs the target:

```python
error = actual_daily_high - forecast_high
```

Important caveat: this dataset is proxy data. Open-Meteo historical forecasts may not preserve exact point-in-time forecast runs visible to a trader, and Open-Meteo actual weather may not match the official Kalshi settlement source.

## Files Created

- `src/weather_data.py`: loaders and validators for actual Open-Meteo weather data.
- `src/forecast_data.py`: loaders and validators for Open-Meteo historical forecast data.
- `scripts/inspect_day5_data.py`: an end-to-end inspection script that loads all four CSVs, validates them, merges daily forecasts with daily actuals, and constructs `error`.

## Import Setup

This cell makes imports work whether the notebook is opened from the repo root or from inside the `notebooks/` directory.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

repo_root

## `src.weather_data`

### `standardize_openmeteo_columns(df, rename_map)`

Renames Open-Meteo columns in a tolerant way. Open-Meteo column names include unit suffixes such as `(F)`, `(%)`, or `(inch)`, and those suffixes can vary slightly across exports or display environments.

This helper checks:

- exact source-column names from `rename_map`
- normalized names with punctuation and unit symbols removed
- base Open-Meteo names before the first parenthesis

For example, it can map a likely max-temperature column to `actual_daily_high` or `forecast_high` even if the unit suffix is a little different.

### `load_hourly_weather(path)`

Loads an Open-Meteo hourly actual-weather CSV with `pd.read_csv(path, skiprows=3)`, because Open-Meteo CSVs include metadata rows above the real header.

It then:

- validates that `time` exists
- converts `time` to pandas datetime
- adds a `date` column from the timestamp
- returns the cleaned DataFrame

### `validate_hourly_weather(df)`

Checks that hourly actual-weather timestamps are safe to use.

It validates:

- `time` exists
- timestamps are unique
- timestamps are monotonic increasing

It also prints a compact profile: shape, date range, columns, and top missing-value percentages.

### `load_daily_weather(path)`

Loads an Open-Meteo daily actual-weather CSV with `skiprows=3`.

It then:

- validates that `time` exists
- converts `time` to a Python date
- adds `date`
- renames max temperature to `actual_daily_high`
- renames min temperature to `actual_daily_low` when present

### `validate_daily_weather(df)`

Checks that daily actual-weather data can support target construction.

It validates:

- `date` exists
- there is one row per date
- `actual_daily_high` exists
- `actual_daily_high >= actual_daily_low` when low temperature is present

It also prints the same compact profile as the hourly validator.

## `src.forecast_data`

The forecast module mirrors the actual-weather module, but uses forecast-specific target names.

### `load_hourly_forecasts(path)`

Loads an Open-Meteo hourly historical-forecast CSV with `skiprows=3`.

It then:

- validates that `time` exists
- converts `time` to pandas datetime
- adds a `date` column

### `validate_hourly_forecasts(df)`

Checks hourly forecast timestamps.

It validates:

- `time` exists
- timestamps are unique
- timestamps are monotonic increasing

It then prints shape, date range, columns, and top missing-value percentages.

### `load_daily_forecasts(path)`

Loads an Open-Meteo daily historical-forecast CSV with `skiprows=3`.

It then:

- validates that `time` exists
- converts `time` to a Python date
- adds `date`
- renames max temperature to `forecast_high`
- renames min temperature to `forecast_low` when present

### `validate_daily_forecasts(df)`

Checks that daily forecast data can be merged with actual weather.

It validates:

- `date` exists
- there is one row per date
- `forecast_high` exists

It then prints shape, date range, columns, and top missing-value percentages.

## Load And Validate The Four Day 5 Files

These paths use the Open-Meteo filenames currently in this repo. The inspection script also supports the shorter filenames from the project spec as primary paths, with these as fallbacks.

In [ ]:
from src.weather_data import (
    load_hourly_weather,
    load_daily_weather,
    validate_hourly_weather,
    validate_daily_weather,
)
from src.forecast_data import (
    load_hourly_forecasts,
    load_daily_forecasts,
    validate_hourly_forecasts,
    validate_daily_forecasts,
)

hourly_weather_path = repo_root / "data/raw/hourly_raw_nyc_openmeteo.csv"
daily_weather_path = repo_root / "data/raw/daily_raw_nyc_openmeteo.csv"
hourly_forecasts_path = repo_root / "data/forecasts/hourly_forecasts_nyc_openmeteo.csv"
daily_forecasts_path = repo_root / "data/forecasts/daily_forecasts_nyc_openmeteo.csv"

hourly_weather = load_hourly_weather(hourly_weather_path)
daily_weather = load_daily_weather(daily_weather_path)
hourly_forecasts = load_hourly_forecasts(hourly_forecasts_path)
daily_forecasts = load_daily_forecasts(daily_forecasts_path)

In [ ]:
validate_hourly_weather(hourly_weather)
validate_daily_weather(daily_weather)
validate_hourly_forecasts(hourly_forecasts)
validate_daily_forecasts(daily_forecasts)

## Construct The Day 5 Target

After validation, target construction is a simple date merge between daily forecasts and daily actual weather.

In [ ]:
merged = daily_forecasts.merge(
    daily_weather,
    on="date",
    how="inner",
    suffixes=("_forecast", "_actual"),
)

merged["error"] = merged["actual_daily_high"] - merged["forecast_high"]

target_columns = ["date", "forecast_high", "actual_daily_high", "error"]
merged[target_columns].head(20)

In [ ]:
merged[target_columns].isna().sum(), merged["error"].describe()

## `scripts.inspect_day5_data`

The script version packages the notebook workflow into one command:

```bash
python scripts/inspect_day5_data.py
```

Important helpers in the script:

- `_resolve_data_path(label, candidates)`: tries the project-spec filename first, then the Open-Meteo filename currently present in this repo.
- `_missing_percentages(df)`: computes missing-value percentages for each column.
- `_print_missing_percentages(df)`: prints nonzero missingness for the merged daily DataFrame.
- `_validate_no_missing(df, column)`: raises an error if a required target column has missing values.
- `_find_column_by_base_name(df, base_name)`: finds columns such as `precipitation_probability_max (%)` from the base name `precipitation_probability_max`.
- `_warn_if_many_missing(df, column, label)`: warns when precipitation-probability columns are too sparse to rely on as core features.
- `main()`: runs the full Day 5 inspection report end to end.

The script validates that merged daily data has no duplicate dates and no missing values in `forecast_high`, `actual_daily_high`, or `error`.

## What Day 5 Gives Us

At the end of Day 5, the project has a clean daily target table with these core columns:

- `date`
- `forecast_high`
- `actual_daily_high`
- `error`

That table is ready for later modeling work, but model training, train/test splitting, and feature engineering are intentionally left out of this notebook.